In [0]:
dbutils.fs.ls("abfss://bronze@grtadlsdev.dfs.core.windows.net/")

In [0]:
# show first 100 rows
df = spark.table("bronze.stops")
display(df.limit(100))

In [0]:
from pyspark.sql import functions as f

# drop useless columns
df = df.drop("stop_desc", "zone_id", "platform_code", "parent_station")

# derive data in col location_type from col stop_url
df = df.withColumn("location_type", f.when(f.col("stop_url").isNull(), "stop").otherwise("station"))

# drop stop_url
df = df.drop("stop_url")

# show first 100 rows
display(df.limit(100))

# check datatypes
df.printSchema()


In [0]:
# change wheelchair boarding to boolean. 1=true, 2=false.
df = df.withColumn("wheelchair_boarding", f.when(f.col("wheelchair_boarding") == 1, f.lit(True)).when(f.col("wheelchair_boarding") == 0, f.lit(False)).otherwise(f.lit(None)))

df.printSchema()

In [0]:
null_counts = df.select([
    f.sum(f.col(c).isNull().cast("int")).alias(c)
    for c in df.columns
]).collect()[0].asDict()

{k: v for k, v in null_counts.items() if v > 0}

In [0]:
df.filter(
    f.col("stop_code").isNull() |
    f.col("wheelchair_boarding").isNull()
).show()

In [0]:
%sql
create schema if not exists silver;

In [0]:
df.write.format("delta").mode("overwrite").option("overwriteSchema" ,"true").saveAsTable("silver.stops")

silver_path = "abfss://silver@grtadlsdev.dfs.core.windows.net/stops/"

# write as delta
df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(silver_path)

display(spark.read.format("delta").load(silver_path).limit(5))
